# 6.2 MoE Inference Lab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshuljain13/llm-inference-at-scale/blob/master/content/07_scaling/06.2_moe_inference/lab.ipynb)
[![Open In Molab](https://raw.githubusercontent.com/marimo-team/marimo/main/docs/_static/marimo-badge.svg)](https://molab.marimo.io/open?url=https://github.com/harshuljain13/llm-inference-at-scale/blob/master/content/07_scaling/06.2_moe_inference/lab.ipynb)

Hands-on experiments: top-k routing, load imbalance, communication volume, cost comparison.

In [ ]:
# Install required packages using subprocess
# This approach works on both Colab and Molab environments
import subprocess, sys
# Run pip install quietly to avoid cluttering output
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'numpy', 'matplotlib'])
# Confirm installation succeeded
print('Dependencies installed.')

In [ ]:
# NumPy provides array operations for routing simulation
import numpy as np
# Matplotlib renders load distribution and cost charts
import matplotlib.pyplot as plt

# === PARAMETERS (change these and re-run) ===
# Number of tokens in the simulated batch
NUM_TOKENS = 1024
# Total expert count in the MoE layer
NUM_EXPERTS = 8
# How many experts each token is routed to
TOP_K = 2
# Seed for reproducible random generation
SEED = 42


def top_k_routing(gate_logits: np.ndarray, k: int) -> tuple:
    """Simulate softmax top-k expert selection.
    Returns (selected_expert_indices, normalized_weights)."""
    # Apply softmax to convert logits to probabilities
    exp_logits = np.exp(gate_logits)
    # Normalize across expert dimension (each token sums to 1)
    probs = exp_logits / exp_logits.sum(axis=1, keepdims=True)
    # Sort and take the k highest-scoring expert indices
    top_k_idx = np.argsort(probs, axis=1)[:, -k:]
    # Extract the probability weights for selected experts
    top_k_w = np.take_along_axis(probs, top_k_idx, axis=1)
    # Renormalize so selected weights sum to 1 per token
    top_k_w = top_k_w / top_k_w.sum(axis=1, keepdims=True)
    return top_k_idx, top_k_w


# Set seed for reproducibility across runs
# === SIMULATION EXECUTION ===
np.random.seed(SEED)
# Generate random router logits (simulates hidden_state -> expert_scores)
gate_logits = np.random.randn(NUM_TOKENS, NUM_EXPERTS)

# Execute top-k routing to get expert assignments
selected_experts, weights = top_k_routing(gate_logits, k=TOP_K)

# Display routing decisions for first 5 tokens
print(f'Tokens: {NUM_TOKENS}, Experts: {NUM_EXPERTS}, Top-K: {TOP_K}')
print(f'First 5 tokens routed to experts:\n{selected_experts[:5]}')
print(f'Corresponding weights:\n{weights[:5].round(3)}')

In [ ]:
def compute_load_stats(selected: np.ndarray, n_experts: int) -> dict:
    """Compute per-expert token counts and imbalance metrics."""
    # Count tokens assigned to each expert (flatten handles top-k)
    load = np.bincount(selected.flatten(), minlength=n_experts)
    # Imbalance: ratio of busiest expert to average (1.0 = perfect)
    imbalance = load.max() / load.mean()
    # Coefficient of variation quantifies spread
    cv = load.std() / load.mean()
    return {'load': load, 'imbalance': imbalance, 'cv': cv}


# Compute load stats for the uniform routing case
stats = compute_load_stats(selected_experts, NUM_EXPERTS)
print(f'Load per expert: {stats["load"]}')
print(f'Imbalance factor (max/mean): {stats["imbalance"]:.3f}')
print(f'CV (coefficient of variation): {stats["cv"]:.3f}')

# Simulate skewed routing where experts 0 and 1 are "hot"
skewed_logits = np.random.randn(NUM_TOKENS, NUM_EXPERTS)
# Add bias to make expert 0 receive disproportionate traffic
skewed_logits[:, 0] += 2.0
# Add smaller bias to expert 1
skewed_logits[:, 1] += 1.5
# Route with the skewed logits
skewed_experts, _ = top_k_routing(skewed_logits, k=TOP_K)
# Compute load stats for the skewed case
skewed_stats = compute_load_stats(skewed_experts, NUM_EXPERTS)

# Create side-by-side comparison plot
# === VISUALIZATION: side-by-side load comparison ===
fig_2, axes_2 = plt.subplots(1, 2, figsize=(12, 4))
for ax_2, s, title, color in zip(
    axes_2,
    [stats, skewed_stats],
    ['Uniform Routing', 'Skewed Routing (Experts 0,1 hot)'],
    ['steelblue', 'coral']
):
    # Bar chart shows token count per expert
    ax_2.bar(range(NUM_EXPERTS), s['load'], color=color, edgecolor='black')
    # Dashed line shows the ideal mean load level
    ax_2.axhline(s['load'].mean(), color='red', linestyle='--',
               label=f'Mean={s["load"].mean():.0f}')
    ax_2.set_xlabel('Expert ID')
    ax_2.set_ylabel('Tokens Assigned')
    ax_2.set_title(f'{title} (imb={s["imbalance"]:.2f})')
    ax_2.legend()
plt.tight_layout()
plt.show()

def double_penalty(load: np.ndarray) -> dict:
    """Calculate compute and memory waste from load imbalance.
    Compute waste: all GPUs wait for the slowest expert GPU.
    Memory waste: buffers sized for worst-case on every GPU."""
    # Mean load represents the ideal balanced case
    mean_l = load.mean()
    # Max load is the bottleneck that all GPUs must wait for
    max_l = load.max()
    # Number of experts (one per GPU in pure EP)
    n = len(load)
    # Compute waste: percentage of idle time on non-bottleneck GPUs
    compute_waste = (max_l - mean_l) / mean_l * 100
    # Memory waste: over-provisioned buffer space across all GPUs
    memory_waste = (max_l * n - load.sum()) / load.sum() * 100
    # Effective utilization (1.0 = perfect, lower = more waste)
    effective_util = mean_l / max_l
    return {
        'compute_waste_pct': compute_waste,
        'memory_waste_pct': memory_waste,
        'effective_util': effective_util
    }


# Show penalty comparison between balanced and skewed routing
print('=== Uniform Routing ===')
for k, v in double_penalty(stats['load']).items():
    print(f'  {k}: {v:.2f}')

print('\n=== Skewed Routing ===')
for k, v in double_penalty(skewed_stats['load']).items():
    print(f'  {k}: {v:.2f}')

# Key insight: skewed routing causes both compute AND memory waste
print('\nSkewed routing wastes GPU cycles AND memory simultaneously.')

In [ ]:
# === PARAMETERS ===
# DeepSeek-V3 hidden dimension
HIDDEN_DIM = 7168
# Tokens in the batch being served
BATCH_TOKENS = 1024
# BF16 = 2 bytes per element
DTYPE_BYTES = 2
# DeepSeek-V3 routes each token to 8 of 256 experts
MOE_TOP_K = 8


def ep_comm_volume_mb(n_tokens, hidden, top_k, n_gpus, dtype=2):
    """Calculate all-to-all communication volume per GPU in MB.
    Accounts for both dispatch (send tokens) and combine (receive results)."""
    # Each GPU processes an equal share of the batch
    tokens_per_gpu = n_tokens // n_gpus
    # Fraction of traffic that must cross GPU boundaries
    # (only 1/n_gpus stays local)
    cross_gpu_frac = (n_gpus - 1) / n_gpus
    # Total bytes = dispatch + combine (x2)
    total_bytes = tokens_per_gpu * top_k * hidden * dtype * cross_gpu_frac * 2
    # Convert bytes to megabytes
    return total_bytes / 1e6


# Sweep over different EP degrees (GPU counts)
# === SWEEP: test different EP degrees ===
gpu_counts = [2, 4, 8, 16, 32, 64]
# Compute communication volume for each configuration
volumes = [ep_comm_volume_mb(BATCH_TOKENS, HIDDEN_DIM, MOE_TOP_K, g, DTYPE_BYTES)
           for g in gpu_counts]

# Print results table
print(f'{"GPUs":>5} {"Comm/GPU (MB)":>15}')
print('-' * 22)
for g, v in zip(gpu_counts, volumes):
    print(f'{g:>5} {v:>15.1f}')

# Visualize communication scaling with GPU count
fig_3, ax_3 = plt.subplots(figsize=(8, 4))
# Log scale on x-axis shows the diminishing per-GPU volume
ax_3.plot(gpu_counts, volumes, 'o-', color='darkblue', linewidth=2, markersize=8)
ax_3.set_xlabel('Number of GPUs (EP degree)')
ax_3.set_ylabel('All-to-All Volume per GPU (MB)')
ax_3.set_title('Expert Parallelism: Communication vs GPU Count')
ax_3.set_xscale('log', base=2)
ax_3.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
# Key insight: per-GPU volume DECREASES with more GPUs
# because each GPU handles fewer local tokens
print('Per-GPU volume decreases but TOTAL cluster traffic increases with EP degree.')

# === PARAMETERS ===
# Approximate cloud cost per GPU-hour (H100 class)
GPU_COST_HR = 3.0

# Model configs: name -> (num_gpus_needed, tokens_per_second_throughput)
configs = {
    'Mixtral 8x7B (MoE)': (2, 2500),
    'Llama-70B (Dense)': (2, 800),
    'Llama-13B (Dense)': (1, 4000),
    'DeepSeek-V3 (MoE)': (32, 4000),
}

# Calculate cost per 1M tokens for each model configuration
print(f'{"Model":<25} {"GPUs":>5} {"Tok/s":>7} {"$/1M tok":>10}')
print('-' * 50)
names, costs = [], []
for name, (gpus, tps) in configs.items():
    # Formula: (GPUs * $/hr) / (tokens/hr) * 1M tokens
    cost_per_m = gpus * GPU_COST_HR / (tps * 3600) * 1e6
    print(f'{name:<25} {gpus:>5} {tps:>7} ${cost_per_m:>8.3f}')
    # Store for plotting
    names.append(name)
    costs.append(cost_per_m)

# Horizontal bar chart comparing costs across models
fig, ax = plt.subplots(figsize=(9, 4))
# Color MoE models coral, dense models blue for visual distinction
colors = ['coral' if 'MoE' in n else 'steelblue' for n in names]
ax.barh(names, costs, color=colors, edgecolor='black')
ax.set_xlabel('Cost per 1M Tokens ($)')
ax.set_title('MoE vs Dense: Serving Cost Comparison')
# Annotate each bar with its dollar value
for i, c in enumerate(costs):
    ax.text(c + 0.2, i, f'${c:.2f}', va='center', fontsize=10)
plt.tight_layout()
plt.show()
# Key takeaway for the reader
print('\nMoE costs more in absolute $ but delivers far more model capacity per dollar.')

In [ ]:
def double_penalty(load: np.ndarray) -> dict:
    """Calculate compute and memory waste from load imbalance.
    Compute waste: all GPUs wait for the slowest expert.
    Memory waste: buffers sized for worst-case on every GPU."""
    mean_l = load.mean()  # ideal load if perfectly balanced
    max_l = load.max()    # actual busiest expert
    n = len(load)         # number of experts
    # Compute waste: fraction of time GPUs idle waiting for max
    compute_waste = (max_l - mean_l) / mean_l * 100  # Compute compute waste
    # Memory waste: buffer_per_expert * n_experts - actual_tokens_used
    memory_waste = (max_l * n - load.sum()) / load.sum() * 100  # Compute memory waste
    # Effective GPU utilization (1.0 = perfect balance)
    effective_util = mean_l / max_l  # Compute effective util
    return {  # Return computed result
        'compute_waste_pct': compute_waste,
        'memory_waste_pct': memory_waste,
        'effective_util': effective_util
    }


# Compare penalties for uniform vs skewed routing
print('=== Uniform Routing ===')  # Display output
for k, v in double_penalty(stats['load']).items():  # Iterate over elements
    print(f'  {k}: {v:.2f}')  # Display output

print('\n=== Skewed Routing ===')  # Display output
for k, v in double_penalty(skewed_stats['load']).items():  # Iterate over elements
    print(f'  {k}: {v:.2f}')  # Display output

## 4. All-to-All Communication Volume

In [ ]:
# === PARAMETERS ===
HIDDEN_DIM = 7168     # DeepSeek-V3 hidden dimension
BATCH_TOKENS = 1024   # tokens in the batch
DTYPE_BYTES = 2       # BF16 = 2 bytes per element
MOE_TOP_K = 8         # DeepSeek-V3 routes to 8 experts


def ep_comm_volume_mb(n_tokens, hidden, top_k, n_gpus, dtype=2):
    """Calculate all-to-all communication volume per GPU in MB.
    Each GPU sends tokens to remote experts (dispatch) and receives results (combine)."""
    # Tokens per GPU (evenly distributed batch)
    tokens_per_gpu = n_tokens // n_gpus
    # Fraction of traffic that crosses GPU boundaries
    cross_gpu_frac = (n_gpus - 1) / n_gpus
    # Total bytes: dispatch + combine (factor of 2)
    total_bytes = tokens_per_gpu * top_k * hidden * dtype * cross_gpu_frac * 2
    return total_bytes / 1e6  # convert to MB


# Sweep over GPU counts to show communication scaling
gpu_counts = [2, 4, 8, 16, 32, 64]
volumes = [ep_comm_volume_mb(BATCH_TOKENS, HIDDEN_DIM, MOE_TOP_K, g, DTYPE_BYTES) for g in gpu_counts]

# Print table of results
print(f'{"GPUs":>5} {"Comm/GPU (MB)":>15}')
for g, v in zip(gpu_counts, volumes):
    print(f'{g:>5} {v:>15.1f}')

# Plot communication volume vs GPU count
fig_5, ax_5 = plt.subplots(figsize=(8, 4))
ax_5.plot(gpu_counts, volumes, 'o-', color='darkblue', linewidth=2, markersize=8)
ax_5.set_xlabel('Number of GPUs (EP degree)')
ax_5.set_ylabel('All-to-All Volume per GPU (MB)')
ax_5.set_title('Expert Parallelism Communication Scaling')
ax_5.set_xscale('log', base=2)
ax_5.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. MoE vs Dense: Cost per Million Tokens

In [ ]:
# === PARAMETERS ===
GPU_COST_HR = 3.0  # $/hr per H100 (approximate cloud pricing)

# Model configs: (num_gpus, tokens_per_second)
configs = {  # Compute configs
    'Mixtral 8x7B (MoE, 2 GPU)': (2, 2500),
    'Llama-70B (Dense, 2 GPU)': (2, 800),
    'Llama-13B (Dense, 1 GPU)': (1, 4000),
    'DeepSeek-V3 (MoE, 32 GPU)': (32, 4000),
}

# Calculate cost per 1M tokens for each config
print(f'{"Model":<30} {"GPUs":>5} {"Tok/s":>7} {"$/1M tok":>10}')  # Display output
print('-' * 55)  # Display output
names, costs = [], []  # Compute names, costs
for name, (gpus, tps) in configs.items():  # Iterate over elements
    # cost = (gpu_count * hourly_rate) / tokens_per_hour * 1M
    cost_per_m = gpus * GPU_COST_HR / (tps * 3600) * 1e6  # Compute cost per m
    print(f'{name:<30} {gpus:>5} {tps:>7} ${cost_per_m:>8.3f}')  # Display output
    names.append(name.split('(')[0].strip())  # Accumulate result
    costs.append(cost_per_m)  # Accumulate result

# Bar chart comparison
fig_6, ax_6 = plt.subplots(figsize=(9, 4))  # Create figure for visualization
colors = ['coral', 'steelblue', 'steelblue', 'coral']  # MoE=coral, Dense=blue
ax_6.barh(names, costs, color=colors, edgecolor='black')  # Compute ax 6.barh(names, costs, color
ax_6.set_xlabel('Cost per 1M Tokens ($)')  # Label x-axis
ax_6.set_title('MoE vs Dense: Serving Cost Comparison')  # Set plot title
# Add value labels on each bar
for i, c in enumerate(costs):  # Iterate over elements
    ax_6.text(c + 0.1, i, f'${c:.2f}', va='center', fontsize=10)  # Format output string
plt.tight_layout()  # Adjust spacing between subplots
plt.show()  # Render the figure
print('\nMoE costs more in absolute $ but delivers far more model capacity per dollar.')  # Display output